# NHANES 2017–2018 patient-need estimate

This notebook estimates a **cross-sectional CKD indicator**, not a confirmed chronic diagnosis. NHANES cannot establish persistence for at least three months. The primary adult (age 18+) rule is race-free 2021 CKD-EPI creatinine eGFR <60 mL/min/1.73 m² **OR** UACR ≥30 mg/g and requires both defining measures. Missing labs are unknown/excluded, never zero. eGFR-only and albuminuria-only definitions are sensitivity analyses.

Population estimates use the NHANES 2-year MEC weight, masked variance strata and PSU, Taylor linearization, Student-t confidence limits, and full-sample domain handling. Raw data come only from the exact official CDC HTTPS files in the committed manifest and are never downloaded in CI.

In [1]:
from dataclasses import asdict
from hashlib import sha256
import json
from pathlib import Path

import pandas as pd

from ckd_intelligence.cohorts.definitions import CKDDefinition, classify_ckd
from ckd_intelligence.statistics.survey import weighted_prevalence

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MANIFEST_PATH = ROOT / 'data/manifests/nhanes-2017-2018-patient-need.json'
RAW_DIR = ROOT / 'data/raw/nhanes/2017-2018'
FIXTURE_PATH = ROOT / 'data/fixtures/nhanes_patient_need_representative.csv'
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))

In [2]:
def checksum(path: Path) -> str:
    return sha256(path.read_bytes()).hexdigest()

expected = {item['name']: item for item in manifest['files']}
has_official_snapshot = all((RAW_DIR / name).is_file() for name in expected)
if has_official_snapshot:
    for name, metadata in expected.items():
        path = RAW_DIR / name
        assert path.stat().st_size == metadata['bytes'], f'Unexpected byte count: {name}'
        assert checksum(path) == metadata['sha256'], f'Checksum mismatch: {name}'
    demo = pd.read_sas(RAW_DIR / 'DEMO_J.xpt', format='xport')[['SEQN', 'RIDAGEYR', 'RIAGENDR', 'WTMEC2YR', 'SDMVSTRA', 'SDMVPSU']]
    biopro = pd.read_sas(RAW_DIR / 'BIOPRO_J.xpt', format='xport')[['SEQN', 'LBXSCR']]
    albumin = pd.read_sas(RAW_DIR / 'ALB_CR_J.xpt', format='xport')[['SEQN', 'URXUMA', 'URXUCR']]
    frame = demo.merge(biopro, on='SEQN', how='left', validate='one_to_one').merge(albumin, on='SEQN', how='left', validate='one_to_one')
    people = frame.rename(columns={'SEQN': 'respondent_id', 'RIDAGEYR': 'age_years', 'LBXSCR': 'serum_creatinine_mg_dl', 'URXUMA': 'urine_albumin_mg_l', 'URXUCR': 'urine_creatinine_mg_dl', 'WTMEC2YR': 'sample_weight', 'SDMVSTRA': 'strata', 'SDMVPSU': 'psu'})
    people['sex'] = frame['RIAGENDR'].map({1.0: 'Male', 2.0: 'Female'})
    people = people.loc[people['sample_weight'].gt(0) & people['strata'].notna() & people['psu'].notna()].copy()
    evidence_type = 'public_observed'
    result_scope = 'NHANES 2017-2018 MEC-exam-weighted U.S. civilian, noninstitutionalized adults'
else:
    people = pd.read_csv(FIXTURE_PATH)
    evidence_type = 'fixture_only'
    result_scope = 'Representative fixture only; not a population estimate'

print({'evidence_type': evidence_type, 'design_rows': len(people), 'result_scope': result_scope})

{'evidence_type': 'public_observed', 'design_rows': 9254, 'result_scope': 'NHANES 2017-2018 MEC-exam-weighted U.S. civilian, noninstitutionalized adults'}


In [3]:
definitions = [CKDDefinition.primary(), CKDDefinition.egfr_only(), CKDDefinition.albuminuria_only()]
for definition in definitions:
    people[definition.name] = classify_ckd(people, definition)

adult = people['age_years'].ge(18)
complete_primary = adult & people[CKDDefinition.primary().name].notna()
waterfall = [
    {'stage_order': 1, 'stage': 'MEC participants with valid design and positive weight', 'people': int(len(people))},
    {'stage_order': 2, 'stage': 'Adults age 18+', 'people': int(adult.sum())},
    {'stage_order': 3, 'stage': 'Complete eGFR and UACR defining labs', 'people': int(complete_primary.sum())},
    {'stage_order': 4, 'stage': 'Cross-sectional CKD indicator positive', 'people': int(people.loc[complete_primary, CKDDefinition.primary().name].sum())},
]
pd.DataFrame(waterfall)

,stage_order,stage,people
0,1,MEC participants with valid design and positiv...,9254
1,2,Adults age 18+,5856
2,3,Complete eGFR and UACR defining labs,5063
3,4,Cross-sectional CKD indicator positive,938


In [4]:
source_population = result_scope
estimates = {}
for definition in definitions:
    estimate = weighted_prevalence(
        people[definition.name].tolist(),
        people['sample_weight'].tolist(),
        people['strata'].tolist(),
        people['psu'].tolist(),
        lonely_psu='error',
        source_population=source_population,
    )
    values = people[definition.name].dropna().astype(bool)
    record = asdict(estimate)
    record['unweighted_point'] = float(values.mean())
    estimates[definition.name] = record

pd.DataFrame(estimates).T[['point', 'standard_error', 'ci_low', 'ci_high', 'denominator', 'sum_weights', 'degrees_freedom', 'unweighted_point']]

,point,standard_error,ci_low,ci_high,denominator,sum_weights,degrees_freedom,unweighted_point
primary_egfr_or_albuminuria,0.137708,0.006463,0.123933,0.151483,5063,230237813.465677,15,0.185266
sensitivity_egfr_only,0.057788,0.004953,0.047231,0.068345,5154,233234204.780414,15,0.085953
sensitivity_albuminuria_only,0.098602,0.00447,0.089074,0.10813,5387,242515116.42351,15,0.136254


In [5]:
primary = estimates['primary_egfr_or_albuminuria']
artifact = {
    'analysis_id': 'nhanes-2017-2018-cross-sectional-ckd-patient-need',
    'evidence_type': evidence_type,
    'result_scope': result_scope,
    'source_release': '2017-2018',
    'source_manifest': str(MANIFEST_PATH.relative_to(ROOT)),
    'input_provenance': manifest['files'] if has_official_snapshot else [{'name': str(FIXTURE_PATH.relative_to(ROOT)), 'sha256': checksum(FIXTURE_PATH)}],
    'definition': {
        'population': 'Adults age 18+ with complete defining labs',
        'primary': 'Race-free CKD-EPI 2021 creatinine eGFR <60 mL/min/1.73 m2 OR UACR >=30 mg/g',
        'missing_labs': 'Unknown and excluded; never false or zero',
        'interpretation': 'Cross-sectional indicator, not confirmed chronic diagnosis',
        'sensitivity': ['eGFR <60 only', 'UACR >=30 mg/g only'],
    },
    'survey_method': {
        'weight': 'WTMEC2YR (2-year MEC exam weight)',
        'strata': 'SDMVSTRA',
        'psu': 'SDMVPSU',
        'variance': 'Taylor linearization for a stratified PSU ratio',
        'confidence_interval': '95% Student-t using PSU-minus-strata design degrees of freedom, bounded to [0,1]',
        'domain_handling': 'Full valid MEC design retained; adults/complete outcomes enter the ratio domain',
        'lonely_psu': 'error',
    },
    'waterfall': waterfall,
    'estimates': estimates,
    'finding': (
        f"Weighted cross-sectional CKD-indicator prevalence was {primary['point']:.1%} "
        f"(95% CI {primary['ci_low']:.1%}–{primary['ci_high']:.1%}; "
        f"n={primary['denominator']:,}) in {result_scope}."
    ),
    'limitations': [
        'One cross-sectional survey cycle cannot confirm chronicity or support diagnosis.',
        'A single serum creatinine and spot UACR are subject to biological and measurement variability.',
        'Estimates describe the documented survey population and should not be interpreted as individual risk.',
    ],
}
output_dir = ROOT / 'data/processed'
output_dir.mkdir(parents=True, exist_ok=True)
artifact_path = output_dir / 'patient_need_summary.json'
artifact_bytes = (json.dumps(artifact, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')
artifact_path.write_bytes(artifact_bytes)
artifact_digest = sha256(artifact_bytes).hexdigest()
(output_dir / 'patient_need_summary.sha256').write_text(f'{artifact_digest}  {artifact_path.name}\n', encoding='utf-8')
print(artifact['finding'])
print(f'Artifact SHA-256: {artifact_digest}')

Weighted cross-sectional CKD-indicator prevalence was 13.8% (95% CI 12.4%–15.1%; n=5,063) in NHANES 2017-2018 MEC-exam-weighted U.S. civilian, noninstitutionalized adults.
Artifact SHA-256: 2aa7fe675e4a85843a4c43bed076ff2a6e3110c2e13dfd010b13806dc708383c


## Interpretation guardrail

The estimate is a population-level screening/need signal for the documented NHANES sample. It is not a diagnosis, treatment recommendation, longitudinal progression estimate, or patient-targeting tool. Sensitivity differences show how much the definition depends on each available marker.